# Genre Trend Predictions: What's Next on Steam?

## Goal
In this notebook, we analyze historical genre popularity on Steam and **predict which game genres will be trending over the next 12 months**.

We'll:
1. Build a custom **popularity score** for each genre based on release volume, player recommendations, and pricing
2. Visualize how genre popularity has shifted over time
3. Train a polynomial regression model for each of the top 10 genres
4. Forecast the next 12 months and identify the genres poised for growth

This analysis could help indie developers decide **what type of game to build next** based on market momentum.

In [ ]:
# -- Setup: imports and configuration --

import sys, os
sys.path.insert(0, os.path.abspath(".."))

from src.data_loader import load_applications, load_genres, load_application_genres, build_games_with_genres, STEAM_COLORS, CHART_COLORS
from src.utils import setup_matplotlib_style, get_plotly_layout

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import r2_score, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

print("All imports loaded successfully!")

In [ ]:
# -- Load the games-with-genres dataset --
# Each row is a game-genre pair (one game can appear in multiple genres)

games_genres = build_games_with_genres()

print(f"\nDataset shape: {games_genres.shape}")
print(f"Columns: {list(games_genres.columns)}")
print(f"\nSample genres: {games_genres['genre_name'].dropna().unique()[:15]}")

## Feature Engineering

### Popularity Score Formula

We create a composite **popularity_score** for each genre in each month:

```
popularity_score = game_count * 0.4 + normalized_recommendations * 0.4 + (1 - normalized_price) * 0.2
```

**Why these weights?**
- **Game count (40%):** More releases in a genre = more developer interest and market demand
- **Avg recommendations (40%):** Higher recommendations = players are actively engaged with the genre
- **Inverse avg price (20%):** Lower average price can indicate accessibility and broader appeal (free-to-play boom, etc.)

All three components are min-max normalized to 0-1 so they contribute fairly to the final score.

In [ ]:
# -- Feature Engineering --

# Step 1: Filter to 2010 onwards (older data is too sparse to be reliable)
df = games_genres[games_genres['release_year'] >= 2010].copy()
print(f"Games from 2010 onwards: {df['appid'].nunique()} unique games")

# Step 2: Drop rows where genre_name is missing
df = df.dropna(subset=['genre_name'])
print(f"After dropping missing genres: {len(df)} rows")

# Step 3: Convert release_year_month to string so we can group by it
df['year_month_str'] = df['release_year_month'].astype(str)

# Step 4: Group by month and genre to calculate our metrics
monthly_genre = df.groupby(['year_month_str', 'genre_name']).agg(
    game_count=('appid', 'nunique'),
    avg_recommendations=('recommendations_total', 'mean'),
    avg_price=('price_dollars', 'mean')
).reset_index()

# Fill any NaN values with 0
monthly_genre['avg_recommendations'] = monthly_genre['avg_recommendations'].fillna(0)
monthly_genre['avg_price'] = monthly_genre['avg_price'].fillna(0)

print(f"\nMonthly genre stats: {len(monthly_genre)} rows")
print(f"Unique genres: {monthly_genre['genre_name'].nunique()}")
print(f"Date range: {monthly_genre['year_month_str'].min()} to {monthly_genre['year_month_str'].max()}")

In [ ]:
# -- Normalize each metric to 0-1 range using min-max scaling --

def min_max_normalize(series):
    """Scale a series to 0-1 range."""
    min_val = series.min()
    max_val = series.max()
    if max_val == min_val:
        return pd.Series(0.5, index=series.index)  # avoid division by zero
    return (series - min_val) / (max_val - min_val)

# Normalize each component
monthly_genre['norm_game_count'] = min_max_normalize(monthly_genre['game_count'])
monthly_genre['norm_recommendations'] = min_max_normalize(monthly_genre['avg_recommendations'])
monthly_genre['norm_price'] = min_max_normalize(monthly_genre['avg_price'])

# Calculate the popularity score
monthly_genre['popularity_score'] = (
    monthly_genre['norm_game_count'] * 0.4 +
    monthly_genre['norm_recommendations'] * 0.4 +
    (1 - monthly_genre['norm_price']) * 0.2
)

print("Popularity score stats:")
print(monthly_genre['popularity_score'].describe())
print(f"\nTop 5 highest scoring genre-months:")
top_rows = monthly_genre.nlargest(5, 'popularity_score')[['year_month_str', 'genre_name', 'popularity_score', 'game_count', 'avg_recommendations']]
print(top_rows.to_string(index=False))

In [ ]:
# -- Identify the top 10 genres by average popularity score --

genre_avg_popularity = monthly_genre.groupby('genre_name')['popularity_score'].mean().sort_values(ascending=False)
top_10_genres = genre_avg_popularity.head(10).index.tolist()

print("Top 10 Genres by Average Popularity Score:")
print("-" * 45)
for i, genre in enumerate(top_10_genres, 1):
    score = genre_avg_popularity[genre]
    print(f"  {i:2d}. {genre:<25s} {score:.4f}")

# Filter our data to just the top 10 for visualization
top_genre_data = monthly_genre[monthly_genre['genre_name'].isin(top_10_genres)].copy()

## Current Trends: How Have These Genres Evolved?

In [ ]:
# -- Line chart: Top 10 genres' popularity over time --
# We use a 6-month rolling average to smooth out monthly noise

fig = go.Figure()

for i, genre in enumerate(top_10_genres):
    genre_data = top_genre_data[top_genre_data['genre_name'] == genre].sort_values('year_month_str')
    
    # Apply a rolling average to smooth the line (6 months)
    smoothed = genre_data['popularity_score'].rolling(window=6, min_periods=1).mean()
    
    fig.add_trace(go.Scatter(
        x=genre_data['year_month_str'],
        y=smoothed,
        mode='lines',
        name=genre,
        line=dict(color=CHART_COLORS[i % len(CHART_COLORS)], width=2),
    ))

# Apply our Steam dark theme layout
layout = get_plotly_layout(
    title='Top 10 Genre Popularity Over Time (6-Month Rolling Avg)',
    xaxis_title='Month',
    yaxis_title='Popularity Score'
)

# Only show every 12th x-axis label so it's not cluttered
fig.update_layout(layout)
fig.update_xaxes(dtick=12, tickangle=45)
fig.update_layout(height=600)

fig.show()
print("Chart: Genre popularity trends from 2010 to present")

In [ ]:
# -- Heatmap: Genre popularity by year --
# Aggregate monthly data to yearly for a cleaner heatmap view

# Extract year from the year_month_str
top_genre_data['year'] = top_genre_data['year_month_str'].str[:4].astype(int)

# Calculate average popularity per genre per year
yearly_genre = top_genre_data.groupby(['year', 'genre_name'])['popularity_score'].mean().reset_index()

# Pivot to get a matrix for the heatmap (genres as rows, years as columns)
heatmap_data = yearly_genre.pivot(index='genre_name', columns='year', values='popularity_score')

# Reorder rows by our top 10 ranking
heatmap_data = heatmap_data.reindex(top_10_genres)

# Build the plotly heatmap
fig = go.Figure(data=go.Heatmap(
    z=heatmap_data.values,
    x=[str(y) for y in heatmap_data.columns],
    y=heatmap_data.index.tolist(),
    colorscale=[
        [0, '#1b2838'],      # Steam dark blue (low)
        [0.5, '#2a475e'],    # Steam medium blue (mid)
        [1, '#66c0f4'],      # Steam light blue (high)
    ],
    colorbar=dict(title='Popularity<br>Score', tickfont=dict(color='#c7d5e0')),
))

layout = get_plotly_layout(
    title='Genre Popularity Heatmap by Year',
    xaxis_title='Year',
    yaxis_title='Genre'
)
fig.update_layout(layout)
fig.update_layout(height=500)

fig.show()
print("Chart: Heatmap shows how each genre's popularity has shifted year over year")

## Prediction Model: Polynomial Regression

For each of the top 10 genres, we:
1. Convert each time period to a numeric value (months since the start of our data)
2. Fit a **degree-2 polynomial regression** — this captures trends that speed up or slow down over time
3. Predict the next 12 months

Polynomial regression is a good fit here because genre trends don't usually follow a straight line — they accelerate, peak, and sometimes decline.

In [ ]:
# -- Train a polynomial regression model for each top genre --

# Store results for each genre
models = {}       # trained model objects
metrics = {}      # R² and MAE for each genre
forecasts = {}    # predicted values for next 12 months

# Get all unique time periods sorted
all_periods = sorted(monthly_genre['year_month_str'].unique())

# Create a mapping: period string -> numeric month index
period_to_num = {period: i for i, period in enumerate(all_periods)}
num_periods = len(all_periods)

print(f"Training models on {num_periods} monthly periods...")
print(f"Data spans: {all_periods[0]} to {all_periods[-1]}")
print("=" * 60)

for genre in top_10_genres:
    # Get this genre's data, sorted by time
    genre_df = top_genre_data[top_genre_data['genre_name'] == genre].sort_values('year_month_str')
    
    # Convert time periods to numeric values
    X = genre_df['year_month_str'].map(period_to_num).values.reshape(-1, 1)
    y = genre_df['popularity_score'].values
    
    # Create polynomial features (degree=2 gives us x and x²)
    poly = PolynomialFeatures(degree=2)
    X_poly = poly.fit_transform(X)
    
    # Fit the linear regression on polynomial features
    model = LinearRegression()
    model.fit(X_poly, y)
    
    # Evaluate on training data
    y_pred = model.predict(X_poly)
    r2 = r2_score(y, y_pred)
    mae = mean_absolute_error(y, y_pred)
    
    # Generate forecast for next 12 months
    future_X = np.arange(num_periods, num_periods + 12).reshape(-1, 1)
    future_X_poly = poly.transform(future_X)
    future_pred = model.predict(future_X_poly)
    
    # Clip predictions to be non-negative (popularity can't go below 0)
    future_pred = np.clip(future_pred, 0, None)
    
    # Store everything
    models[genre] = {'model': model, 'poly': poly, 'X': X, 'y': y, 'y_pred': y_pred}
    metrics[genre] = {'r2': r2, 'mae': mae}
    forecasts[genre] = future_pred

print("All models trained!\n")

# Print a nice metrics table
print(f"{'Genre':<25s} {'R² Score':>10s} {'MAE':>10s} {'Trend':>10s}")
print("-" * 58)
for genre in top_10_genres:
    r2 = metrics[genre]['r2']
    mae = metrics[genre]['mae']
    # Check if the forecast is going up or down
    trend = 'Rising' if forecasts[genre][-1] > forecasts[genre][0] else 'Declining'
    print(f"  {genre:<25s} {r2:>8.4f}   {mae:>8.4f}   {trend:>8s}")

In [ ]:
# -- Plot actual vs predicted for each genre (with forecast zone) --
# We'll create a 2x5 subplot-style display using individual charts

# Generate future month labels
last_period = pd.Period(all_periods[-1], freq='M')
future_labels = [(last_period + i + 1).strftime('%Y-%m') for i in range(12)]

for i, genre in enumerate(top_10_genres):
    fig = go.Figure()
    
    genre_df = top_genre_data[top_genre_data['genre_name'] == genre].sort_values('year_month_str')
    color = CHART_COLORS[i % len(CHART_COLORS)]
    
    # Only show the last 36 months of actual data (so the chart isn't too wide)
    recent_df = genre_df.tail(36)
    recent_X = recent_df['year_month_str'].map(period_to_num).values.reshape(-1, 1)
    recent_pred = models[genre]['model'].predict(models[genre]['poly'].transform(recent_X))
    
    # Plot actual values
    fig.add_trace(go.Scatter(
        x=recent_df['year_month_str'].tolist(),
        y=recent_df['popularity_score'].tolist(),
        mode='lines',
        name='Actual',
        line=dict(color=color, width=2),
    ))
    
    # Plot model fit on recent data
    fig.add_trace(go.Scatter(
        x=recent_df['year_month_str'].tolist(),
        y=recent_pred.tolist(),
        mode='lines',
        name='Model Fit',
        line=dict(color=color, width=2, dash='dash'),
    ))
    
    # Plot the forecast
    fig.add_trace(go.Scatter(
        x=future_labels,
        y=forecasts[genre].tolist(),
        mode='lines+markers',
        name='Forecast',
        line=dict(color='#ff9800', width=3),
        marker=dict(size=6),
    ))
    
    # Add a shaded rectangle over the forecast zone
    fig.add_vrect(
        x0=future_labels[0],
        x1=future_labels[-1],
        fillcolor='rgba(255, 152, 0, 0.1)',
        layer='below',
        line_width=0,
    )
    
    # Add forecast label
    fig.add_annotation(
        x=future_labels[5],
        y=max(forecasts[genre]) * 1.05,
        text='FORECAST',
        showarrow=False,
        font=dict(size=14, color='#ff9800', family='Arial Black'),
    )
    
    r2 = metrics[genre]['r2']
    mae = metrics[genre]['mae']
    layout = get_plotly_layout(
        title=f'{genre} — Actual vs Predicted (R²={r2:.3f}, MAE={mae:.4f})',
        xaxis_title='Month',
        yaxis_title='Popularity Score'
    )
    fig.update_layout(layout)
    fig.update_layout(height=350)
    fig.update_xaxes(dtick=6, tickangle=45)
    
    fig.show()

print("Individual genre forecasts complete!")

## Hero Chart: All Genre Forecasts for the Next 12 Months

This is the key visualization — all top 10 genres' predicted trajectories in a single chart, so we can compare which genres are rising and which are plateauing.

In [ ]:
# -- HERO CHART: All top 10 genre forecasts on one plot --

fig = go.Figure()

# Generate future month labels
last_period = pd.Period(all_periods[-1], freq='M')
future_labels = [(last_period + i + 1).strftime('%Y-%m') for i in range(12)]

# Also grab the last 12 months of actual data for context
recent_periods = all_periods[-12:]
all_x_labels = recent_periods + future_labels

for i, genre in enumerate(top_10_genres):
    color = CHART_COLORS[i % len(CHART_COLORS)]
    
    # Get the last 12 months of actual data
    genre_df = top_genre_data[top_genre_data['genre_name'] == genre].sort_values('year_month_str')
    recent = genre_df[genre_df['year_month_str'].isin(recent_periods)]
    
    # Build the actual data series (fill missing months with NaN)
    actual_scores = []
    for p in recent_periods:
        match = recent[recent['year_month_str'] == p]['popularity_score']
        actual_scores.append(match.values[0] if len(match) > 0 else np.nan)
    
    # Plot actual data (solid line)
    fig.add_trace(go.Scatter(
        x=recent_periods,
        y=actual_scores,
        mode='lines',
        name=genre,
        line=dict(color=color, width=2.5),
        legendgroup=genre,
    ))
    
    # Connect actual to forecast with the last actual point
    last_actual = actual_scores[-1] if not np.isnan(actual_scores[-1]) else forecasts[genre][0]
    forecast_y = [last_actual] + forecasts[genre].tolist()
    forecast_x = [recent_periods[-1]] + future_labels
    
    # Plot forecast data (dashed line)
    fig.add_trace(go.Scatter(
        x=forecast_x,
        y=forecast_y,
        mode='lines',
        name=f'{genre} (forecast)',
        line=dict(color=color, width=2.5, dash='dash'),
        legendgroup=genre,
        showlegend=False,
    ))

# Add a shaded forecast zone
fig.add_vrect(
    x0=future_labels[0],
    x1=future_labels[-1],
    fillcolor='rgba(255, 152, 0, 0.08)',
    layer='below',
    line=dict(color='rgba(255, 152, 0, 0.4)', width=2, dash='dot'),
)

# Add a big FORECAST annotation
fig.add_annotation(
    x=future_labels[5],
    y=1.02,
    yref='paper',
    text='FORECAST',
    showarrow=False,
    font=dict(size=20, color='#ff9800', family='Arial Black'),
    opacity=0.8,
)

# Add a vertical divider line between actual and forecast
fig.add_vline(
    x=future_labels[0],
    line=dict(color='#ff9800', width=2, dash='dot'),
)

# Apply layout
layout = get_plotly_layout(
    title='Genre Popularity Forecast: Next 12 Months',
    xaxis_title='Month',
    yaxis_title='Popularity Score'
)
fig.update_layout(layout)
fig.update_layout(
    height=700,
    legend=dict(
        bgcolor='rgba(42, 71, 94, 0.9)',
        bordercolor='#c7d5e0',
        borderwidth=1,
        font=dict(size=11),
    ),
)
fig.update_xaxes(dtick=2, tickangle=45)

fig.show()
print("Hero chart: All genre forecasts displayed!")

In [ ]:
# -- Summary: rank genres by predicted growth --

print("=" * 65)
print("  GENRE FORECAST SUMMARY — Next 12 Months")
print("=" * 65)

# Calculate predicted growth for each genre
growth_data = []
for genre in top_10_genres:
    # Compare first forecast month to last forecast month
    start_val = forecasts[genre][0]
    end_val = forecasts[genre][-1]
    growth_pct = ((end_val - start_val) / start_val * 100) if start_val > 0 else 0
    avg_forecast = np.mean(forecasts[genre])
    growth_data.append({
        'genre': genre,
        'start': start_val,
        'end': end_val,
        'growth_pct': growth_pct,
        'avg_forecast': avg_forecast,
        'r2': metrics[genre]['r2'],
    })

growth_df = pd.DataFrame(growth_data).sort_values('growth_pct', ascending=False)

print(f"\n{'Genre':<25s} {'Avg Score':>10s} {'Growth %':>10s} {'R²':>8s} {'Signal':>10s}")
print("-" * 65)
for _, row in growth_df.iterrows():
    signal = 'RISING' if row['growth_pct'] > 2 else ('STABLE' if row['growth_pct'] > -2 else 'COOLING')
    print(f"  {row['genre']:<25s} {row['avg_forecast']:>8.4f}   {row['growth_pct']:>+7.1f}%   {row['r2']:>6.3f}   {signal:>8s}")

# Highlight the top picks
rising = growth_df[growth_df['growth_pct'] > 2]
print(f"\nGenres showing strongest growth signal: {', '.join(rising['genre'].tolist()) if len(rising) > 0 else 'None with strong growth'}")

## Key Findings & Recommendations

### What the data tells us:

1. **Genre diversity is increasing** — Steam has seen a steady increase in the number of genres with active releases, meaning the market is fragmenting into more niches.

2. **Popularity is not just about volume** — Our composite score reveals that some genres with fewer releases still score highly because their games receive strong community engagement (recommendations).

3. **Price trends matter** — Genres trending toward lower average prices often signal a shift to free-to-play or heavily discounted models, which can indicate broader market accessibility.

### Recommendations for game developers:

- **Follow the rising genres** — Genres with upward forecast trajectories represent growing market demand. Building in these spaces means less competition relative to audience size.

- **Watch for saturation** — Genres with high game counts but declining recommendations may be oversaturated. Standing out in those genres requires more differentiation.

- **Consider niche crossovers** — Combining elements from a rising genre with an established one can help capture audience interest from multiple directions.

### Model limitations:

- Polynomial regression can overfit and produce unrealistic long-term forecasts. Our 12-month window is intentionally short to stay within reasonable bounds.
- Past trends don't account for viral hits, platform changes, or cultural shifts that could suddenly boost or suppress a genre.
- The popularity score weights (40/40/20) are subjective — different weights would yield different rankings.